# Collecte des données (suite) #
>
>## II - Données astronomiques ##
>>
>>Nous avons précédemment collecté les données de **production d'énergie solaire**, contenant la base de calcul de notre variable cible. Or, les autres variables du jeu de données collecté ne permettent à priori pas d'expliquer la variable cible.
>>
>> - Elles ne contiennent pas de données sur la **position du soleil** par rapport aux panneaux solaires ;
>> - Elles ne contiennent pas de données **météorologiques**.
>>
>>Pour déterminer quelles données pourraient permettre d'expliquer notre variable cible et pourraient être utiles à notre modélisation, nous sommes partis des connaissances usuelles de l'homme du métier (voir les articles retenus).
>>
>>Parmi ces connaissances, on a constaté que la position du soleil par rapport à l'emplacement des panneaux solaires était importante et que cette position était déterminée par l'**azimut** et l'**angle solaire**.
>>
>>On a vu que la région PACA était vaste et on a déterminé **cinq lieux significatifs** pour la production d'énergie solaire : nous allons calculer l'azimut et l'angle solaire pour chacun d'eux.

>>### A - Récupération des données de production ###
>>>
>>>Nous commencons par récupérer les données de production collectées à l'étape précédente depuis le Drive du projet.

In [ ]:
# On se donne accès au Drive qui contient les fichiers sources
from google.colab import drive

# Monter Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Chemin pour récupérer le dataset de production
chemin_dataset_production = '/content/drive/MyDrive/Flux/00_Collecte_datasets/01_Production/' + 'production_2020_2024.csv'

# Chemin vers le répertoire de travail pour les datasets de données astronomiques
chemin_datasets_astro = '/content/drive/MyDrive/Flux/00_Collecte_datasets/02_Astronomie/'


>>>On importe les librairies nécessaires pour manipuler nos données :

In [ ]:
# On importe les librairies dont on a besoin pour traiter les datasets
import pandas as pd
import numpy as np

>>>On charge le jeu de données de l'étape précédente :

In [ ]:
# Charger le dataset production final de l'étape précédente :
df_rte = pd.read_csv(chemin_dataset_production, parse_dates=['datetime_utc'])
display(df_rte.head())
df_rte.dtypes

/tmp/ipython-input-231226874.py:2: DtypeWarning: Columns (6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_rte = pd.read_csv(chemin_dataset_production, parse_dates=['datetime_utc'])


,datetime_utc,Périmètre,Nature,Consommation,Solaire,Ech. physiques,Stockage batterie,Déstockage batterie,TCO Solaire (%),TCH Solaire (%)
0,2019-12-31 23:00:00+00:00,PACA,Données définitives,6123.0,0.0,3332.0,-,-,0.0,0.0
1,2019-12-31 23:30:00+00:00,PACA,Données définitives,5907.0,0.0,2837.0,-,-,0.0,0.0
2,2020-01-01 00:00:00+00:00,PACA,Données définitives,5724.0,0.0,2668.0,-,-,0.0,0.0
3,2020-01-01 00:30:00+00:00,PACA,Données définitives,5749.0,0.0,2754.0,-,-,0.0,0.0
4,2020-01-01 01:00:00+00:00,PACA,Données définitives,5700.0,0.0,2886.0,-,-,0.0,0.0


,0
datetime_utc,"datetime64[ns, UTC]"
Périmètre,object
Nature,object
Consommation,float64
Solaire,float64
Ech. physiques,float64
Stockage batterie,object
Déstockage batterie,object
TCO Solaire (%),float64
TCH Solaire (%),float64


>>### B - Lieux retenus ###
>>>
>>>On récupère les coordonnées des cinq points significatifs que l'on a précédemment déterminé.
>>>
>>>Pour faciliter l'exploitation ultérieure des données géographiques, on remplace le nom de villes sélectionnées par un code sur trois lettres :    
>>>
>>> - Cruis est encodé par 'CRU' ;
>>> - Saint-Étienne-le-Laus par 'SEL' ;
>>> - Saint-Vallier-de-Thiey par 'SVT' ;
>>> - Bras par 'BRA' ;
>>> - Eygalières par 'EYG'.

In [ ]:
# On a retenu 5 lieux pertinents pour l'étude de production d'énergie solaire
# On stocke leur code ainsi que leur coordonnées géographiques (latitude et longitude) dans un dictionnaire
lieux_retenus = {'CRU' : {'latitude': 44.0845, 'longitude': 5.8397},
                 'SEL' : {'latitude': 44.5075, 'longitude': 6.1616},
                 'SVT' : {'latitude': 43.6994, 'longitude': 6.8516},
                 'BRA' : {'latitude': 43.4723, 'longitude': 5.9558},
                 'EYG' : {'latitude': 43.7638, 'longitude': 4.9554}}

>>### C - Calcul de l'azimut et de l'altitude ###
>>>
>>>Des passionnés d'astronomie ont déjà mis à dispositions plusieurs librairies permettant de calculer l'azimut et l'altitude du soleil (= angle solaire).
>>>
>>>On va utiliser la librairie PySolar : https://pysolar.readthedocs.io/en/latest/.


>>>On commence par l'installer :

In [ ]:
# On installe la librairie pysolar
!pip install pysolar

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.5 MB/s eta 0:00:00


>>>Puis on l'importe :

In [ ]:
# On importe les librairies dont on a besoin
from pysolar.solar import *
import datetime

>>>Enfin on utilise la librairie pysolar pour calculer l'azimuth et l'angle solaire au niveau des cinq points significatifs retenus pour chaque observation de notre jeu de données.

In [ ]:
# Attention, l'exécution de cette cellule est longue !
# On parcours les villes
for ville, location in lieux_retenus.items() :
    print(ville + "...")
    latitude = location['latitude']
    longitude = location['longitude']

    # On crée une nouvelle colonne pour la ville pour l'azimuth
    print("\tAzimuth...")
    df_rte[ville + '_azimuth'] = df_rte['datetime_utc'].apply(lambda date : get_azimuth(latitude, longitude, date))
    print("\t\tOK")

    # On crée une nouvelle colonne pour la ville pour l'altitude
    print("\tAltitude...")
    df_rte[ville + '_altitude'] = df_rte['datetime_utc'].apply(lambda date : get_altitude(latitude, longitude, date))
    print("\t\tOK")

    print(ville + "-> OK")


CRU...
	Azimuth...
		OK
	Altitude...
		OK
CRU-> OK
SEL...
	Azimuth...
		OK
	Altitude...
		OK
SEL-> OK
SVT...
	Azimuth...
		OK
	Altitude...
		OK
SVT-> OK
BRA...
	Azimuth...
		OK
	Altitude...
		OK
BRA-> OK
EYG...
	Azimuth...
		OK
	Altitude...
		OK
EYG-> OK


>>>On affiche le début du jeu de données pour avoir une première visualisation des nouvelles données :

In [ ]:
display(df_rte.head(10))

,datetime_utc,Périmètre,Nature,Consommation,Solaire,Ech. physiques,Stockage batterie,Déstockage batterie,TCO Solaire (%),TCH Solaire (%),CRU_azimuth,CRU_altitude,SEL_azimuth,SEL_altitude,SVT_azimuth,SVT_altitude,BRA_azimuth,BRA_altitude,EYG_azimuth,EYG_altitude
0,2019-12-31 23:00:00+00:00,PACA,Données définitives,6123.0,0.0,3332.0,-,-,0.0,0.0,335.596867,-67.454606,336.734433,-67.160776,337.561520,-68.097845,335.241040,-68.046190,333.265411,-67.469007
1,2019-12-31 23:30:00+00:00,PACA,Données définitives,5907.0,0.0,2837.0,-,-,0.0,0.0,353.817769,-68.883102,354.738150,-68.485180,356.336005,-69.329403,353.945722,-69.500739,351.451534,-69.119526
2,2020-01-01 00:00:00+00:00,PACA,Données définitives,5724.0,0.0,2668.0,-,-,0.0,0.0,12.883689,-68.564280,13.430816,-68.099948,15.637215,-68.757908,13.536657,-69.141123,10.834111,-69.009045
3,2020-01-01 00:30:00+00:00,PACA,Données définitives,5749.0,0.0,2754.0,-,-,0.0,0.0,30.261125,-66.570892,30.451783,-66.089459,32.859299,-66.517353,31.240402,-67.054165,28.719513,-67.163582
4,2020-01-01 01:00:00+00:00,PACA,Données définitives,5700.0,0.0,2886.0,-,-,0.0,0.0,44.631373,-63.284229,44.594428,-62.821307,46.883423,-63.030485,45.699703,-63.656435,43.545408,-63.957709
5,2020-01-01 01:30:00+00:00,PACA,Données définitives,5803.0,0.0,3157.0,-,-,0.0,0.0,56.104984,-59.129358,55.961399,-58.701947,58.001012,-58.726843,57.122520,-59.396257,55.332468,-59.836248
6,2020-01-01 02:00:00+00:00,PACA,Données définitives,5566.0,0.0,3761.0,-,-,0.0,0.0,65.346368,-54.428472,65.171692,-54.042195,66.944253,-53.917283,66.258196,-54.602951,64.764317,-55.141210
7,2020-01-01 02:30:00+00:00,PACA,Données définitives,5433.0,0.0,3922.0,-,-,0.0,0.0,73.008055,-49.393849,72.842496,-49.049410,74.372636,-48.802125,73.801849,-49.488368,72.533553,-50.097161
8,2020-01-01 03:00:00+00:00,PACA,Données définitives,5332.0,0.0,3878.0,-,-,0.0,0.0,79.584354,-44.161912,79.447862,-43.858394,80.769679,-43.509092,80.264541,-44.186485,79.165262,-44.847029
9,2020-01-01 03:30:00+00:00,PACA,Données définitives,5219.0,0.0,3751.0,-,-,0.0,0.0,85.424299,-38.823177,85.326319,-38.559425,86.472049,-38.123002,86.000566,-38.785359,85.027894,-39.484567


>>### D - Enregistrement des données obtenues de Pysolar ###
>>>
>>>Nous avons terminé la collecte des données explicatives relatives à l'`azimuth` et à l'`angle solaire` pour chaque point significatif du point de vue de la production d'énergie solaire, grâce à la librairie **PySolar**.
>>>
>>>Nous enregistrons notre jeu de données actuel pour clore la deuxième partie de notre travail de collecte.

In [ ]:
# On enregistre ce dataset contenant les données de production et les données astronomiques avant ajout d'autres variables
df_rte.to_csv(chemin_datasets_astro + "astro_2020_2024.csv", index=False)

In [ ]:
# Exemple de la manière dont charger ce dataset production+astro final :
df = pd.read_csv(chemin_datasets_astro + "astro_2020_2024.csv", parse_dates=['datetime_utc'])
display(df.head())
df.dtypes

/tmp/ipython-input-270453828.py:2: DtypeWarning: Columns (6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(chemin_datasets_astro + "astro_2020_2024.csv", parse_dates=['datetime_utc'])


,datetime_utc,Périmètre,Nature,Consommation,Solaire,Ech. physiques,Stockage batterie,Déstockage batterie,TCO Solaire (%),TCH Solaire (%),CRU_azimuth,CRU_altitude,SEL_azimuth,SEL_altitude,SVT_azimuth,SVT_altitude,BRA_azimuth,BRA_altitude,EYG_azimuth,EYG_altitude
0,2019-12-31 23:00:00+00:00,PACA,Données définitives,6123.0,0.0,3332.0,-,-,0.0,0.0,335.596867,-67.454606,336.734433,-67.160776,337.561520,-68.097845,335.241040,-68.046190,333.265411,-67.469007
1,2019-12-31 23:30:00+00:00,PACA,Données définitives,5907.0,0.0,2837.0,-,-,0.0,0.0,353.817769,-68.883102,354.738150,-68.485180,356.336005,-69.329403,353.945722,-69.500739,351.451534,-69.119526
2,2020-01-01 00:00:00+00:00,PACA,Données définitives,5724.0,0.0,2668.0,-,-,0.0,0.0,12.883689,-68.564280,13.430816,-68.099948,15.637215,-68.757908,13.536657,-69.141123,10.834111,-69.009045
3,2020-01-01 00:30:00+00:00,PACA,Données définitives,5749.0,0.0,2754.0,-,-,0.0,0.0,30.261125,-66.570892,30.451783,-66.089459,32.859299,-66.517353,31.240402,-67.054165,28.719513,-67.163582
4,2020-01-01 01:00:00+00:00,PACA,Données définitives,5700.0,0.0,2886.0,-,-,0.0,0.0,44.631373,-63.284229,44.594428,-62.821307,46.883423,-63.030485,45.699703,-63.656435,43.545408,-63.957709


,0
datetime_utc,"datetime64[ns, UTC]"
Périmètre,object
Nature,object
Consommation,float64
Solaire,float64
Ech. physiques,float64
Stockage batterie,object
Déstockage batterie,object
TCO Solaire (%),float64
TCH Solaire (%),float64


In [ ]:
df.describe()

,Consommation,Solaire,Ech. physiques,TCO Solaire (%),TCH Solaire (%),CRU_azimuth,CRU_altitude,SEL_azimuth,SEL_altitude,SVT_azimuth,SVT_altitude,BRA_azimuth,BRA_altitude,EYG_azimuth,EYG_altitude
count,87696.000000,87696.000000,87696.000000,87696.000000,87696.000000,87696.000000,87696.000000,87696.000000,87696.000000,87696.000000,87696.000000,87696.000000,87696.000000,87696.000000,87696.000000
mean,4485.721048,301.790241,2362.531826,6.700169,16.873232,180.414164,0.332711,180.600596,0.334114,180.383371,0.331374,180.481003,0.330811,179.903427,0.331568
std,855.835873,438.892803,862.312299,9.822056,23.957124,101.362763,34.060740,101.415049,33.843022,101.319389,34.258665,101.285925,34.375236,101.325082,34.225587
min,1640.000000,0.000000,-615.000000,0.000000,0.000000,0.017255,-69.332104,0.001426,-68.919902,0.005349,-69.739151,0.027417,-69.947758,0.000959,-69.606933
25%,3852.000000,0.000000,1793.000000,0.000000,0.000000,90.222951,-25.625496,90.314566,-25.403531,90.048959,-25.862663,90.310370,-25.957515,89.958512,-25.700124
50%,4379.000000,3.000000,2351.000000,0.080000,0.200000,180.009202,0.614024,179.996787,0.677727,180.004222,0.668943,179.993561,0.563709,179.997015,0.621216
75%,5051.000000,559.000000,2895.000000,12.180000,32.320000,270.809415,26.241474,270.866820,25.979124,270.771208,26.416424,270.793477,26.594034,270.350968,26.453837
max,8044.000000,1942.000000,5657.000000,62.320000,91.350000,359.990157,69.288364,359.969741,68.886867,359.995370,69.724723,359.996740,69.905504,359.983575,69.533977
